In [ ]:
import requests
from datetime import datetime
Api_key = "PpknPk8zxSfOPh10Ll4yqpAC7AmpDYJ1cVbnYlmf"
Astriods_data = []
Api_url = f'https://api.nasa.gov/neo/rest/v1/feed?start_date=2025-01-01&end_date=2025-01-08&api_key={Api_key}'
while len(Astriods_data) < 10000:
    Astro_data = requests.get(Api_url).json()
    Required_data = Astro_data['near_earth_objects']
    for details,info in Required_data.items():
        for data in info:
            Astriods_data.append(dict(id=data['id'],
            neo_reference_id=data['neo_reference_id'],
            name=data['name'],
            absolute_magnitude_h=data['absolute_magnitude_h'],
            estimated_diameter_min=data['estimated_diameter']['kilometers']['estimated_diameter_min'],
            estimated_diameter_max=data['estimated_diameter']['kilometers']['estimated_diameter_max'],
            is_potentially_hazardous_asteroid=data['is_potentially_hazardous_asteroid'],
            close_approach_date=datetime.strptime((data['close_approach_data'][0]['close_approach_date']),"%Y-%m-%d"),
            relative_velocity_kmph=float(data['close_approach_data'][0]['relative_velocity']['kilometers_per_hour']),
            astronomical=float(data['close_approach_data'][0]['miss_distance']['astronomical']),
            miss_distance_km=float(data['close_approach_data'][0]['miss_distance']['kilometers']),
            miss_distance_lunar=float(data['close_approach_data'][0]['miss_distance']['lunar']),
            orbiting_body=data['close_approach_data'][0]['orbiting_body']))
            if len(Astriods_data) >= 10000:
                break
        if len(Astriods_data) >= 10000:
            break                        
Api_url = Astro_data['links'].get('next')

In [ ]:
print(Astriods_data)

[{'id': '2226514', 'neo_reference_id': '2226514', 'name': '226514 (2003 UX34)', 'absolute_magnitude_h': 20.14, 'estimated_diameter_min': 0.2492039814, 'estimated_diameter_max': 0.5572370428, 'is_potentially_hazardous_asteroid': True, 'close_approach_date': datetime.datetime(2025, 1, 7, 0, 0), 'relative_velocity_kmph': 59576.37781234, 'astronomical': 0.1236081659, 'miss_distance_km': 18491518.333246633, 'miss_distance_lunar': 48.0835765351, 'orbiting_body': 'Earth'}, {'id': '2438017', 'neo_reference_id': '2438017', 'name': '438017 (2003 YO3)', 'absolute_magnitude_h': 18.54, 'estimated_diameter_min': 0.5206609142, 'estimated_diameter_max': 1.1642331974, 'is_potentially_hazardous_asteroid': False, 'close_approach_date': datetime.datetime(2025, 1, 7, 0, 0), 'relative_velocity_kmph': 54996.3055018069, 'astronomical': 0.0811263309, 'miss_distance_km': 12136326.303555183, 'miss_distance_lunar': 31.5581427201, 'orbiting_body': 'Earth'}, {'id': '2481442', 'neo_reference_id': '2481442', 'name': 

In [3]:
import mysql.connector as db
import pandas as pd
connection=db.connect(host='localhost',
                      user='root',
                      password='Mithran@21',
                      database='training')
cursur=connection.cursor()

In [ ]:
table = pd.DataFrame(Astriods_data)

In [ ]:
cursur.execute("""create table asteroids(
               id int,
               name varchar(100),
               absolute_magnitude_h float,
               estimated_diameter_min float,
               estimated_diameter_max float,
               is_potentially_hazardous_asteroid boolean);""")
connection.commit()

In [ ]:
insert="""insert into asteroids values(%s,%s,%s,%s,%s,%s)"""
for i in range(len(table)):
    data = (int(table['id'][i]),
            str(table['name'][i]),
            float(table['absolute_magnitude_h'][i]),
            float(table['estimated_diameter_min'][i]),
            float(table['estimated_diameter_max'][i]),
            bool(table['is_potentially_hazardous_asteroid'][i]))
    cursur.execute(insert,data)
connection.commit()

In [ ]:
cursur.execute("""create table close_approach(neo_reference_id int,
               close_approach_date date,
               relative_velocity_kmph float,
               astronomical float,
               miss_distance_km float,
               miss_distance_lunar float,
               orbiting_body varchar(100));""")
connection.commit()


In [ ]:
insert="""insert into close_approach values(%s,%s,%s,%s,%s,%s,%s)"""
for i in range(len(table)):
    data = (int(table['neo_reference_id'][i]),
            table['close_approach_date'][i].date(),
            float(table['relative_velocity_kmph'][i]),
            float(table['astronomical'][i]),
            float(table['miss_distance_km'][i]),
            float(table['miss_distance_lunar'][i]),
            str(table['orbiting_body'][i]))
    cursur.execute(insert,data)
connection.commit()

In [ ]:
cursur.execute('''select name,count(name) as 'count' from training.asteroids
               group by name;''') 
astroids_count=cursur.fetchall()
astroids_count_table=pd.DataFrame(astroids_count,columns=['name','count'])
print(astroids_count_table)

                   name  count
0    226514 (2003 UX34)    234
1     438017 (2003 YO3)    234
2     481442 (2006 WO3)    234
3           (2010 AL60)    234
4            (2015 NU2)    234
..                  ...    ...
124          (2024 BM1)    231
125         (2024 YA10)    231
126           (2025 AD)    231
127           (2025 AT)    231
128          (2025 AO3)    231

[129 rows x 2 columns]


In [ ]:
cursur.execute('''select asteroids.name,
               AVG(close_approach.relative_velocity_kmph) as "avg_velocity"
               from training.asteroids
               inner join training.close_approach 
               on asteroids.id = close_approach.neo_reference_id
               group by asteroids.name
               order by avg_velocity desc;''')
asteroids_velocity=cursur.fetchall()
asteroids_velocity_table=pd.DataFrame(asteroids_velocity,columns=['name','avg_velocity'])
print(asteroids_velocity_table)

                   name   avg_velocity
0            (2020 PP5)  126807.304688
1    494999 (2010 JU39)  111054.953125
2            (2024 YZ4)   99517.390625
3           (2016 BD15)   93383.367188
4             (2015 BF)   92433.554688
..                  ...            ...
124          (2024 YY4)   14430.696289
125          (2024 YR3)   13304.285156
126         (2021 VH35)   11892.868164
127         (2024 XB20)   10221.601562
128          (2025 AG1)    9522.503906

[129 rows x 2 columns]


In [ ]:
Fastest_astroid=asteroids_velocity_table.loc[0:9]
print(Fastest_astroid)

                 name   avg_velocity
0          (2020 PP5)  126807.304688
1  494999 (2010 JU39)  111054.953125
2          (2024 YZ4)   99517.390625
3         (2016 BD15)   93383.367188
4           (2015 BF)   92433.554688
5          (2022 EQ6)   87773.148438
6         (2019 AO12)   87401.554688
7          (2020 QL7)   85415.273438
8          (2020 WR1)   82488.500000
9          (2019 AR8)   82067.929688


In [ ]:
cursur.execute('''select name,count(name) as 'name_count'
               from training.asteroids
               where is_potentially_hazardous_asteroid = 1 
               group by name
               having name_count > 3;''')
Danger_astro=cursur.fetchall()


danger_astro_table=pd.DataFrame(Danger_astro,columns=['name','Danger_Asteroid_count'])
print(danger_astro_table)

                 name  Danger_Asteroid_count
0  226514 (2003 UX34)                    234
1          (2015 NU2)                    234
2         (2016 BD15)                    234
3          (2020 BC6)                    234
4  494999 (2010 JU39)                    231
5   451003 (2008 UD1)                    231


In [ ]:
cursur.execute('''select monthname(close_approach_date) as month,
               count(*) as month_count
               from training.close_approach
               group by monthname(close_approach_date)
               order by month_count desc
               limit 1;''')
Max_approach_month=cursur.fetchall()
Max_approach_month_table=pd.DataFrame(Max_approach_month,columns=['month','month_count'])
print(Max_approach_month_table)

     month  month_count
0  January        10000


In [ ]:
cursur.execute('''select asteroids.name,close_approach.relative_velocity_kmph
                from training.asteroids
                inner join training.close_approach
                on asteroids.id = close_approach.neo_reference_id
                order by close_approach.relative_velocity_kmph desc
                limit 1;''')
Max_velocity=cursur.fetchall()
Max_velocity_table=pd.DataFrame(Max_velocity,columns=['name','Max_velocity'])
print(Max_velocity_table)

         name  Max_velocity
0  (2020 PP5)      126807.0


In [ ]:
cursur.execute('''select name,estimated_diameter_max 
               from training.asteroids
               order by estimated_diameter_max desc;''')
Maximum_diameter=cursur.fetchall()
Maximum_diameter_table=pd.DataFrame(Maximum_diameter,columns=['name','Maximum_diameter'])   
print(Maximum_diameter_table)

                       name  Maximum_diameter
0      887 Alinda (A918 AA)         10.281100
1      887 Alinda (A918 AA)         10.281100
2      887 Alinda (A918 AA)         10.281100
3      887 Alinda (A918 AA)         10.281100
4      887 Alinda (A918 AA)         10.281100
...                     ...               ...
29995            (2025 AG1)          0.007517
29996            (2025 AG1)          0.007517
29997            (2025 AG1)          0.007517
29998            (2025 AG1)          0.007517
29999            (2025 AG1)          0.007517

[30000 rows x 2 columns]


In [ ]:
cursur.execute('''select asteroids.name,close_approach.close_approach_date,close_approach.miss_distance_km
                from training.asteroids 
               inner join training.close_approach
               on asteroids.id = close_approach.neo_reference_id
               order by close_approach.miss_distance_km desc;''')
Astroids_comenear_earth=cursur.fetchall()
Astroids_comenear_earth_table=pd.DataFrame(Astroids_comenear_earth,columns=['name','close_approach_date','miss_distance_km'])
print(Astroids_comenear_earth_table)

               name close_approach_date  miss_distance_km
0        (2020 PP5)          2025-01-08        73850600.0
1        (2020 PP5)          2025-01-08        73850600.0
2        (2020 PP5)          2025-01-08        73850600.0
3        (2020 PP5)          2025-01-08        73850600.0
4        (2020 PP5)          2025-01-08        73850600.0
...             ...                 ...               ...
2325673   (2025 AC)          2025-01-02          140221.0
2325674   (2025 AC)          2025-01-02          140221.0
2325675   (2025 AC)          2025-01-02          140221.0
2325676   (2025 AC)          2025-01-02          140221.0
2325677   (2025 AC)          2025-01-02          140221.0

[2325678 rows x 3 columns]


In [ ]:
cursur.execute('''select asteroids.name,close_approach.close_approach_date,close_approach.miss_distance_km
                from training.asteroids 
               inner join training.close_approach
               on asteroids.id = close_approach.neo_reference_id
               order by asteroids.name;''')
Astroids_comenear_data=cursur.fetchall()
Astroids_comenear_data_table=pd.DataFrame(Astroids_comenear_data,columns=['name','close_approach_date','miss_distance_km'])
print(Astroids_comenear_data_table)

                         name close_approach_date  miss_distance_km
0                 (2010 AL30)          2025-01-08        19541500.0
1                 (2010 AL30)          2025-01-08        19541500.0
2                 (2010 AL30)          2025-01-08        19541500.0
3                 (2010 AL30)          2025-01-08        19541500.0
4                 (2010 AL30)          2025-01-08        19541500.0
...                       ...                 ...               ...
2325673  887 Alinda (A918 AA)          2025-01-08        12296600.0
2325674  887 Alinda (A918 AA)          2025-01-08        12296600.0
2325675  887 Alinda (A918 AA)          2025-01-08        12296600.0
2325676  887 Alinda (A918 AA)          2025-01-08        12296600.0
2325677  887 Alinda (A918 AA)          2025-01-08        12296600.0

[2325678 rows x 3 columns]


In [ ]:
cursur.execute('''select asteroids.name,close_approach.relative_velocity_kmph
               from training.asteroids 
               inner join training.close_approach
               on asteroids.id = close_approach.neo_reference_id
               where close_approach.relative_velocity_kmph > 50000
               order by close_approach.relative_velocity_kmph desc;''')
High_velocity=cursur.fetchall()
High_velocity_table=pd.DataFrame(High_velocity,columns=['name','High_velocity'])
print(High_velocity_table)

                     name  High_velocity
0              (2020 PP5)       126807.0
1              (2020 PP5)       126807.0
2              (2020 PP5)       126807.0
3              (2020 PP5)       126807.0
4              (2020 PP5)       126807.0
...                   ...            ...
866326  451003 (2008 UD1)        50939.1
866327  451003 (2008 UD1)        50939.1
866328  451003 (2008 UD1)        50939.1
866329  451003 (2008 UD1)        50939.1
866330  451003 (2008 UD1)        50939.1

[866331 rows x 2 columns]


In [ ]:
cursur.execute('''select monthname(close_approach_date) as month,
               count(*) as month_count
               from training.close_approach
               group by monthname(close_approach_date)
               order by month_count desc;''')
Max_approach_monthcount=cursur.fetchall()
Max_approach_monthcount_table=pd.DataFrame(Max_approach_monthcount,columns=['month','month_count'])
print(Max_approach_monthcount_table)

     month  month_count
0  January        10000


In [ ]:
cursur.execute('''select asteroids.name,min(asteroids.absolute_magnitude_h) as 'min_absolute_magnitude_h'
               from training.asteroids
               group by asteroids.name
               order by min_absolute_magnitude_h
               limit 1;''')
Min_absolute_magnitude=cursur.fetchall()
Min_absolute_magnitude_table=pd.DataFrame(Min_absolute_magnitude,columns=['name','min_absolute_magnitude_h'])
print(Min_absolute_magnitude_table)

                   name  min_absolute_magnitude_h
0  887 Alinda (A918 AA)                     13.81


In [ ]:
cursur.execute('''select asteroids.name,
               count(case when asteroids.is_potentially_hazardous_asteroid = 1 then 1 end) as 'harzardous_count',
               count(case when asteroids.is_potentially_hazardous_asteroid = 0 then 1 end) as 'non_hazardous_count'
               from training.asteroids
              group by asteroids.name;''')
Hazardous_asteroids=cursur.fetchall()   
Hazardous_asteroids_table=pd.DataFrame(Hazardous_asteroids,columns=['name','hazardous_count','non_hazardous_count'])
print(Hazardous_asteroids_table)

                   name  hazardous_count  non_hazardous_count
0    226514 (2003 UX34)              234                    0
1     438017 (2003 YO3)                0                  234
2     481442 (2006 WO3)                0                  234
3           (2010 AL60)                0                  234
4            (2015 NU2)              234                    0
..                  ...              ...                  ...
124          (2024 BM1)                0                  231
125         (2024 YA10)                0                  231
126           (2025 AD)                0                  231
127           (2025 AT)                0                  231
128          (2025 AO3)                0                  231

[129 rows x 3 columns]


In [ ]:
cursur.execute('''select asteroids.name,close_approach.miss_distance_km,close_approach.close_approach_date
                from training.asteroids 
                inner join training.close_approach
                on asteroids.id = close_approach.neo_reference_id
                where close_approach.miss_distance_km < 384400;''')
Astriods_lessthan_oneLD=cursur.fetchall()
Astriods_lessthan_oneLD_table=pd.DataFrame(Astriods_lessthan_oneLD,columns=['name','miss_distance_km','close_approach_date'])   
print(Astriods_lessthan_oneLD_table)

            name  miss_distance_km close_approach_date
0      (2025 AB)          153127.0          2025-01-03
1      (2025 AB)          153127.0          2025-01-03
2      (2025 AB)          153127.0          2025-01-03
3      (2025 AB)          153127.0          2025-01-03
4      (2025 AB)          153127.0          2025-01-03
...          ...               ...                 ...
36034  (2025 AB)          153127.0          2025-01-03
36035  (2025 AB)          153127.0          2025-01-03
36036  (2025 AB)          153127.0          2025-01-03
36037  (2025 AB)          153127.0          2025-01-03
36038  (2025 AB)          153127.0          2025-01-03

[36039 rows x 3 columns]


In [ ]:
cursur.execute('''select asteroids.name,close_approach.miss_distance_km
                from training.asteroids 
                inner join training.close_approach
                on asteroids.id = close_approach.neo_reference_id
                where close_approach.miss_distance_km < 7479894;''')
Astriods_lessthan_0_05au=cursur.fetchall()
Astriods_lessthan_0_05au_table=pd.DataFrame(Astriods_lessthan_0_05au,columns=['name','miss_distance_km'])   
print(Astriods_lessthan_0_05au_table)

              name  miss_distance_km
0       (2024 YZ9)         3623220.0
1       (2024 YZ9)         3623220.0
2       (2024 YZ9)         3623220.0
3       (2024 YZ9)         3623220.0
4       (2024 YZ9)         3623220.0
...            ...               ...
649162   (2025 AE)         6288670.0
649163   (2025 AE)         6288670.0
649164   (2025 AE)         6288670.0
649165   (2025 AE)         6288670.0
649166   (2025 AE)         6288670.0

[649167 rows x 2 columns]


In [6]:
cursur.execute('''select Max(close_approach_date) as 'max_date'
                from training.close_approach;''')
Max_date=cursur.fetchall()
print(Max_date)

[(datetime.date(2025, 1, 8),)]


In [11]:
cursur.execute('''select * from training.asteroids
                order by is_potentially_hazardous_asteroid;''')
is_potentially_hazardous_asteroid_filter=cursur.fetchall()
is_potentially_hazardous_asteroid_filter_table=pd.DataFrame(is_potentially_hazardous_asteroid_filter,columns=['id','name','absolute_magnitude_h','estimated_diameter_min','estimated_diameter_max','is_potentially_hazardous_asteroid'])
print(is_potentially_hazardous_asteroid_filter_table)


             id                name  absolute_magnitude_h  \
0      54516261           (2025 BS)                 23.64   
1       3553148         (2010 XB11)                 19.69   
2       3645041         (2013 NC15)                 22.14   
3       3647191          (2013 RT9)                 26.50   
4       3837864         (2019 AS11)                 26.80   
...         ...                 ...                   ...   
29995   2451003   451003 (2008 UD1)                 19.49   
29996   2226514  226514 (2003 UX34)                 20.14   
29997   3723888          (2015 NU2)                 20.91   
29998   3740934         (2016 BD15)                 21.20   
29999   3989144          (2020 BC6)                 20.85   

       estimated_diameter_min  estimated_diameter_max  \
0                    0.049723                0.111183   
1                    0.306588                0.685551   
2                    0.099210                0.221840   
3                    0.013322          

In [15]:
cursur.execute('''select * from training.asteroids
               order by estimated_diameter_min;''')
estimated_diameter_min_filter=cursur.fetchall()
estimated_diameter_min_filter_table=pd.DataFrame(estimated_diameter_min_filter,columns=['id','name','absolute_magnitude_h','estimated_diameter_min','estimated_diameter_max','is_potentially_hazardous_asteroid'])  
print(estimated_diameter_min_filter_table)

             id                  name  absolute_magnitude_h  \
0      54514249            (2025 AG1)                 29.49   
1      54514249            (2025 AG1)                 29.49   
2      54514249            (2025 AG1)                 29.49   
3      54514249            (2025 AG1)                 29.49   
4      54514249            (2025 AG1)                 29.49   
...         ...                   ...                   ...   
29995   2000887  887 Alinda (A918 AA)                 13.81   
29996   2000887  887 Alinda (A918 AA)                 13.81   
29997   2000887  887 Alinda (A918 AA)                 13.81   
29998   2000887  887 Alinda (A918 AA)                 13.81   
29999   2000887  887 Alinda (A918 AA)                 13.81   

       estimated_diameter_min  estimated_diameter_max  \
0                    0.003362                0.007517   
1                    0.003362                0.007517   
2                    0.003362                0.007517   
3              

In [6]:
cursur.execute('''select asteroids.name,close_approach.miss_distance_lunar
                from training.asteroids 
                inner join training.close_approach
                on asteroids.id = close_approach.neo_reference_id
                order by close_approach.miss_distance_lunar;''')
Astroids_earth_lunar=cursur.fetchall()
Astroids_earth_lunar_table=pd.DataFrame(Astroids_earth_lunar,columns=['name','miss_distance_lunar'])
print(Astroids_earth_lunar_table)

               name  miss_distance_lunar
0         (2025 AC)             0.364618
1         (2025 AC)             0.364618
2         (2025 AC)             0.364618
3         (2025 AC)             0.364618
4         (2025 AC)             0.364618
...             ...                  ...
2325673  (2020 PP5)           192.034000
2325674  (2020 PP5)           192.034000
2325675  (2020 PP5)           192.034000
2325676  (2020 PP5)           192.034000
2325677  (2020 PP5)           192.034000

[2325678 rows x 2 columns]


In [8]:
cursur.execute('''select asteroids.name,count(is_potentially_hazardous_asteroid) as 'count'
                from training.asteroids
                inner join training.close_approach
                on asteroids.id = close_approach.neo_reference_id
                group by asteroids.name,is_potentially_hazardous_asteroid;''')
Hazardous_asteroids_count=cursur.fetchall() 
Hazardous_asteroids_count_table=pd.DataFrame(Hazardous_asteroids_count,columns=['name','count'])
print(Hazardous_asteroids_count_table)


                   name  count
0    226514 (2003 UX34)  18252
1     438017 (2003 YO3)  18252
2     481442 (2006 WO3)  18252
3           (2010 AL60)  18252
4            (2015 NU2)  18252
..                  ...    ...
124          (2024 BM1)  17787
125         (2024 YA10)  17787
126           (2025 AD)  17787
127           (2025 AT)  17787
128          (2025 AO3)  17787

[129 rows x 2 columns]


In [11]:
cursur.execute('''select asteroids.name,asteroids.is_potentially_hazardous_asteroid,close_approach.miss_distance_km,close_approach.close_approach_date
                from training.asteroids 
                inner join training.close_approach
                on asteroids.id = close_approach.neo_reference_id
                where asteroids.is_potentially_hazardous_asteroid = 1
               and close_approach.miss_distance_km < 5000000
               order by close_approach.miss_distance_km;''')
Hazardous_asteroids_miss_distance=cursur.fetchall()
Hazardous_asteroids_miss_distance_table=pd.DataFrame(Hazardous_asteroids_miss_distance,columns=['name','is_potentially_hazardous_asteroid','miss_distance_km','close_approach_date'])
print(Hazardous_asteroids_miss_distance_table)         

             name  is_potentially_hazardous_asteroid  miss_distance_km  \
0      (2020 BC6)                                  1         3683380.0   
1      (2020 BC6)                                  1         3683380.0   
2      (2020 BC6)                                  1         3683380.0   
3      (2020 BC6)                                  1         3683380.0   
4      (2020 BC6)                                  1         3683380.0   
...           ...                                ...               ...   
18247  (2020 BC6)                                  1         3683380.0   
18248  (2020 BC6)                                  1         3683380.0   
18249  (2020 BC6)                                  1         3683380.0   
18250  (2020 BC6)                                  1         3683380.0   
18251  (2020 BC6)                                  1         3683380.0   

      close_approach_date  
0              2025-01-05  
1              2025-01-05  
2              2025-01-05  